# Sicilian NMT with NLLB-200 — zero-shot + **bidirectional** LoRA fine-tune

Evaluate **NLLB-200** zero-shot, then **LoRA fine-tune in BOTH directions** (scn↔en) and
re-evaluate. Bidirectional training fixes the weak en→scn direction and can help scn→en.
Default model `nllb-200-1.3B`; LoRA r=32 on q/k/v/out_proj. Sicilian = `scn_Latn`.

**Steps:** Runtime → **GPU**. Run top to bottom; upload the 6 data files via the left Files
panel into `/content` if Drive isn't synced. Artifacts save to Drive (and sync to your PC).
Bidirectional fine-tune is ~2× the data → a few hours on a T4; keep the tab active.

Best so far (scn→en, frozen test): NLLB-1.3B fine-tuned **31.16 BLEU** (above the paper baseline 29.1).

In [1]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao   # Colab's old torchao breaks peft's LoRA import

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00


In [2]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/SicilianNMT-colab'
except Exception:
    OUT = 'sicilian-nllb-out'
os.makedirs(OUT, exist_ok=True)
print('using', OUT)

using sicilian-nllb-out


In [3]:
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
need = [f'{s}.{l}' for s in ('train', 'valid', 'test') for l in ('scn', 'en')]
if all(os.path.exists(f'{DATA}/{f}') for f in need):
    base = DATA; print('reading data from Drive:', DATA)
elif all(os.path.exists(f) for f in need):
    base = '.'; print('reading data from /content')
else:
    print('Upload the 6 files via the left Files panel, then re-run this cell.')
    from google.colab import files; files.upload(); base = '.'
splits = {s: {l: read(f'{base}/{s}.{l}') for l in ('scn', 'en')} for s in ('train', 'valid', 'test')}
print({s: len(d['scn']) for s, d in splits.items()})

Upload the 6 files via the left Files panel, then re-run this cell.


Saving itsc.it to itsc.it
Saving itsc.scn to itsc.scn
Saving itsc_test.it to itsc_test.it
Saving itsc_test.scn to itsc_test.scn
Saving itsc_train.it to itsc_train.it
Saving itsc_train.scn to itsc_train.scn
Saving itsc_valid.it to itsc_valid.it
Saving itsc_valid.scn to itsc_valid.scn
Saving test.en to test.en
Saving test.scn to test.scn
Saving test.tsv to test.tsv
Saving train.en to train.en
Saving train.scn to train.scn
Saving train.tsv to train.tsv
Saving valid.en to valid.en
Saving valid.scn to valid.scn
Saving valid.tsv to valid.tsv
{'train': 27386, 'valid': 1000, 'test': 1000}


In [4]:
import os, gc, json, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # set before any CUDA alloc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF

# free any model left on the GPU from a previous run, so re-running this cell never stacks two
for _v in ('model', 'ft', 'base'):
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL = 'facebook/nllb-200-1.3B'   # or 'facebook/nllb-200-distilled-600M'
LANG = {'scn': 'scn_Latn', 'en': 'eng_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('WARNING: NO GPU. Runtime -> Change runtime type -> GPU, then Restart and run all.')
print('loading', MODEL, '...')
tok = AutoTokenizer.from_pretrained(MODEL)
# fp16 on GPU: the 1.3B in fp32 + beam search OOMs a 14.5GB T4
dtype = torch.float16 if device == 'cuda' else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=dtype, low_cpu_mem_usage=True).to(device).eval()
results = {}

def translate(texts, src, tgt, bs=8, max_len=160):
    tok.src_lang = LANG[src]
    tgt_id = tok.convert_tokens_to_ids(LANG[tgt])
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  translated {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print()
    torch.cuda.empty_cache()
    return out

def report(hyp, ref, tag):
    b = BLEU().corpus_score(hyp, [ref]).score
    c = CHRF().corpus_score(hyp, [ref]).score
    results[tag] = {'bleu': round(b, 2), 'chrf': round(c, 2)}
    fname = tag.replace(' ', '_').replace('->', '2')
    open(f'{OUT}/hyp_{fname}.txt', 'w', encoding='utf-8').write('\n'.join(hyp) + '\n')
    json.dump(results, open(f'{OUT}/results.json', 'w'), indent=2, ensure_ascii=False)
    print(f'{tag}:  BLEU={b:.2f}  chrF={c:.2f}   (saved to {OUT})')
print('ready on', device, '|', dtype)

loading facebook/nllb-200-1.3B ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

ready on cuda | torch.float16


In [5]:
# ---- ZERO-SHOT (both directions) ----
report(translate(splits['test']['scn'], 'scn', 'en'), splits['test']['en'],  'zero-shot scn->en')
report(translate(splits['test']['en'],  'en', 'scn'), splits['test']['scn'], 'zero-shot en->scn')

  translated 1000/1000
zero-shot scn->en:  BLEU=29.02  chrF=55.23   (saved to sicilian-nllb-out)
  translated 1000/1000
zero-shot en->scn:  BLEU=9.89  chrF=41.12   (saved to sicilian-nllb-out)


## Bidirectional LoRA fine-tune (scn↔en)

If it OOMs on a T4, drop `per_device_train_batch_size` to 2 (and raise grad-accum to 8).

In [6]:
from datasets import Dataset, concatenate_datasets
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

torch.cuda.empty_cache()
EPOCHS = 2   # bidirectional => the data is doubled

ft = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',
                                      target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
                                      task_type='SEQ_2_SEQ_LM'))
# base stays fp16 (frozen); keep the trainable LoRA params in fp32 for stable fp16 training
for p in ft.parameters():
    if p.requires_grad:
        p.data = p.data.float()
ft.config.use_cache = False
ft.enable_input_require_grads()
ft.print_trainable_parameters()

def tok_dir(src_list, tgt_list, src_lang, tgt_lang):
    tok.src_lang, tok.tgt_lang = src_lang, tgt_lang
    return Dataset.from_dict({'src': src_list, 'tgt': tgt_list}).map(
        lambda b: tok(b['src'], text_target=b['tgt'], max_length=128, truncation=True),
        batched=True, remove_columns=['src', 'tgt'])

ds_s2e = tok_dir(splits['train']['scn'], splits['train']['en'], 'scn_Latn', 'eng_Latn')
ds_e2s = tok_dir(splits['train']['en'], splits['train']['scn'], 'eng_Latn', 'scn_Latn')
train_ds = concatenate_datasets([ds_s2e, ds_e2s]).shuffle(seed=13)
print('bidirectional training examples:', len(train_ds))

args = Seq2SeqTrainingArguments(
    output_dir=f'{OUT}/trainer-bidir-1.3B', num_train_epochs=EPOCHS,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=2e-4, fp16=True, logging_steps=100, save_strategy='no', report_to=[])
Seq2SeqTrainer(model=ft, args=args, train_dataset=train_ds,
               data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()

trainable params: 18,874,368 || all params: 1,389,512,704 || trainable%: 1.3583


Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

bidirectional training examples: 54772


Step,Training Loss
100,8.674115
200,8.164600
300,8.136141
400,7.859540
500,7.777108
600,7.559813
700,7.960459
800,7.593633
900,7.568839
1000,7.615304


Step,Training Loss
100,8.674115
200,8.164600
300,8.136141
400,7.859540
500,7.777108
600,7.559813
700,7.960459
800,7.593633
900,7.568839
1000,7.615304


TrainOutput(global_step=6848, training_loss=7.303177846926395, metrics={'train_runtime': 21022.7108, 'train_samples_per_second': 5.211, 'train_steps_per_second': 0.326, 'total_flos': 3.678837335064576e+16, 'train_loss': 7.303177846926395, 'epoch': 2.0})

In [7]:
# ---- EVALUATE FINE-TUNED (both directions) + SAVE ADAPTER ----
model = ft.eval(); model.config.use_cache = True
report(translate(splits['test']['scn'], 'scn', 'en'), splits['test']['en'],  'LoRA bidir scn->en 1.3B')
report(translate(splits['test']['en'],  'en', 'scn'), splits['test']['scn'], 'LoRA bidir en->scn 1.3B')
ft.save_pretrained(f'{OUT}/nllb-lora-bidir-1.3B')
print('saved adapter + results.json + hyps to', OUT)
print(json.dumps(results, indent=2, ensure_ascii=False))

  translated 1000/1000
LoRA bidir scn->en 1.3B:  BLEU=31.43  chrF=56.94   (saved to sicilian-nllb-out)
  translated 1000/1000
LoRA bidir en->scn 1.3B:  BLEU=18.73  chrF=49.96   (saved to sicilian-nllb-out)
saved adapter + results.json + hyps to sicilian-nllb-out
{
  "zero-shot scn->en": {
    "bleu": 29.02,
    "chrf": 55.23
  },
  "zero-shot en->scn": {
    "bleu": 9.89,
    "chrf": 41.12
  },
  "LoRA bidir scn->en 1.3B": {
    "bleu": 31.43,
    "chrf": 56.94
  },
  "LoRA bidir en->scn 1.3B": {
    "bleu": 18.73,
    "chrf": 49.96
  }
}


Artifacts under `MyDrive/SicilianNMT-colab/` sync to your PC. NLLB numbers are raw text
(compare to the raw floor 5.27); the Sockeye 9.79 was tokenized space.